## Organization

In [10]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()
os.environ["TAVILY_API_KEY"] 
os.environ["OPENAI_API_KEY"] 
os.environ["LANGCHAIN_TRACING_V2"] 
os.environ["LANGCHAIN_PROJECT"] 
os.environ["LANGCHAIN_API_KEY"] 

'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'

In [11]:
from openai import OpenAI
from langsmith.wrappers import wrap_openai
from langsmith import traceable

openai_client = wrap_openai(OpenAI())

## Task 1:  Parse Notebook

Python implementation equivalent to the TypeScript notebookParser.ts

In [12]:
import json
import re
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass, field, asdict
import hashlib


@dataclass
class CodeNode:
    """Represents a code element in the notebook."""
    id: str
    type: str  # 'cell' | 'class' | 'function' | 'import' | 'variable'
    name: str
    content: str
    cell_index: int
    line_in_cell: int
    line_in_notebook: int
    children: List['CodeNode'] = field(default_factory=list)
    dependencies: List[str] = field(default_factory=list)
    metadata: Optional[Dict[str, Any]] = None


@dataclass
class GraphData:
    """Contains nodes and edges for the code dependency graph."""
    nodes: List[CodeNode] = field(default_factory=list)
    edges: List[Dict[str, str]] = field(default_factory=list)


class NotebookParser:
    """Parser for Jupyter notebooks to extract code structure and dependencies."""
    
    def __init__(self):
        self.node_id = 0
    
    def parse_notebook(self, notebook_data: Dict[str, Any]) -> GraphData:
        """Parse a notebook from JSON data (similar to vscode.NotebookDocument)."""
        nodes = []
        edges = []
        absolute_line = 0
        
        if 'cells' not in notebook_data:
            raise ValueError('Invalid notebook format: missing cells')
        
        for cell_index, cell in enumerate(notebook_data['cells']):
            if cell.get('cell_type') == 'code':
                cell_node = self._parse_cell(cell, cell_index, absolute_line)
                nodes.append(cell_node)
                
                code_content = self._get_cell_source(cell)
                child_nodes = self._parse_code_structure(code_content, cell_index, absolute_line)
                cell_node.children = child_nodes
                nodes.extend(self._flatten_nodes(child_nodes))
            
            # Update absolute line counter
            content = self._get_cell_source(cell)
            absolute_line += len(content.split('\n')) + 1
        
        self._detect_dependencies(nodes, edges)
        
        return GraphData(nodes=nodes, edges=edges)
    
    def parse(self, notebook_json: str) -> GraphData:
        """Legacy method for JSON string parsing (backward compatibility)."""
        notebook_data = json.loads(notebook_json)
        return self.parse_notebook(notebook_data)
    
    def _parse_cell(self, cell: Dict[str, Any], index: int, absolute_line: int) -> CodeNode:
        """Parse a single notebook cell."""
        content = self._get_cell_source(cell)
        
        return CodeNode(
            id=f'cell_{index}',
            type='cell',
            name=f'Cell {index + 1}',
            content=content,
            cell_index=index,
            line_in_cell=0,
            line_in_notebook=absolute_line,
            metadata=cell.get('metadata')
        )
    
    def _get_cell_source(self, cell: Dict[str, Any]) -> str:
        """Extract source code from a cell."""
        source = cell.get('source', '')
        if isinstance(source, list):
            return ''.join(source)
        return source
    
    def _parse_code_structure(self, code: str, cell_index: int, cell_start_line: int) -> List[CodeNode]:
        """Parse Python code structure to extract classes, functions, imports, and variables."""
        nodes = []
        lines = code.split('\n')
        
        # Regex patterns for Python code elements
        import_regex = re.compile(r'^(?:from\s+(\S+)\s+)?import\s+(.+)$')
        class_regex = re.compile(r'^class\s+(\w+)(?:\s*\(([^)]*)\))?\s*:')
        function_regex = re.compile(r'^def\s+(\w+)\s*\(([^)]*)\)\s*(?:->\s*[^:]+)?\s*:')
        variable_regex = re.compile(r'^(\w+)\s*=\s*(.+)$')
        
        current_indent = 0
        parent_stack = []
        
        for i, line in enumerate(lines):
            trimmed_line = line.strip()
            
            if not trimmed_line or trimmed_line.startswith('#'):
                continue
            
            # Calculate indentation
            indent = len(line) - len(line.lstrip())
            if not line.strip():
                continue
            
            # Pop from parent stack if we've decreased indentation
            while parent_stack and indent <= current_indent:
                parent_stack.pop()
                current_indent = parent_stack[-1].line_in_cell if parent_stack else 0
            
            node = None
            
            # Check for imports
            if import_match := import_regex.match(trimmed_line):
                node = CodeNode(
                    id=f'import_{self.node_id}',
                    type='import',
                    name=import_match.group(1) or import_match.group(2),
                    content=trimmed_line,
                    cell_index=cell_index,
                    line_in_cell=i,
                    line_in_notebook=cell_start_line + i
                )
                self.node_id += 1
            
            # Check for classes
            elif class_match := class_regex.match(trimmed_line):
                node = CodeNode(
                    id=f'class_{class_match.group(1)}_{self.node_id}',
                    type='class',
                    name=class_match.group(1),
                    content=trimmed_line,
                    cell_index=cell_index,
                    line_in_cell=i,
                    line_in_notebook=cell_start_line + i
                )
                self.node_id += 1
            
            # Check for functions
            elif function_match := function_regex.match(trimmed_line):
                node = CodeNode(
                    id=f'function_{function_match.group(1)}_{self.node_id}',
                    type='function',
                    name=function_match.group(1),
                    content=trimmed_line,
                    cell_index=cell_index,
                    line_in_cell=i,
                    line_in_notebook=cell_start_line + i
                )
                self.node_id += 1
            
            # Check for top-level variables
            elif indent == 0 and (var_match := variable_regex.match(trimmed_line)):
                node = CodeNode(
                    id=f'var_{var_match.group(1)}_{self.node_id}',
                    type='variable',
                    name=var_match.group(1),
                    content=trimmed_line,
                    cell_index=cell_index,
                    line_in_cell=i,
                    line_in_notebook=cell_start_line + i
                )
                self.node_id += 1
            
            if node:
                if parent_stack:
                    parent_stack[-1].children.append(node)
                else:
                    nodes.append(node)
                
                if node.type in ['class', 'function']:
                    parent_stack.append(node)
                    current_indent = indent
        
        return nodes
    
    def _flatten_nodes(self, nodes: List[CodeNode]) -> List[CodeNode]:
        """Flatten nested node structure into a flat list."""
        flattened = []
        
        for node in nodes:
            flattened.append(node)
            if node.children:
                flattened.extend(self._flatten_nodes(node.children))
        
        return flattened
    
    def _detect_dependencies(self, nodes: List[CodeNode], edges: List[Dict[str, str]]):
        """Detect dependencies between code nodes."""
        # Create a map of node names to nodes
        node_map = {}
        for node in nodes:
            if node.type != 'cell':
                node_map[node.name] = node
        
        # Look for function/class calls in function and class bodies
        for node in nodes:
            if node.type in ['function', 'class']:
                # Find all function calls in the content
                call_pattern = re.compile(r'\b(\w+)\s*\(')
                matches = call_pattern.findall(node.content)
                
                for called_name in matches:
                    if called_name in node_map and called_name != node.name:
                        edges.append({
                            'source': node.id,
                            'target': node_map[called_name].id,
                            'type': 'calls'
                        })
                        node.dependencies.append(called_name)


# Helper functions for serialization
def node_to_dict(node: CodeNode) -> Dict[str, Any]:
    """Convert a CodeNode to a dictionary."""
    return {
        'id': node.id,
        'type': node.type,
        'name': node.name,
        'content': node.content,
        'cellIndex': node.cell_index,
        'lineInCell': node.line_in_cell,
        'lineInNotebook': node.line_in_notebook,
        'children': [node_to_dict(child) for child in node.children],
        'dependencies': node.dependencies,
        'metadata': node.metadata
    }


def graph_to_dict(graph: GraphData) -> Dict[str, Any]:
    """Convert GraphData to a dictionary."""
    return {
        'nodes': [node_to_dict(node) for node in graph.nodes],
        'edges': graph.edges
    }

## Test Implementation with Tiny Demo Notebook

Based on the chatgpt.md requirements, let's create a tiny demo notebook and test our implementation

In [13]:
# Create a tiny demo notebook structure similar to what chatgpt.md suggests
tiny_demo_notebook = {
    "cells": [
        {
            "cell_type": "markdown",
            "source": ["# Demo RAG Application\n", "This notebook demonstrates a simple RAG implementation."]
        },
        {
            "cell_type": "code",
            "source": [
                "# Import necessary libraries\n",
                "import pandas as pd\n",
                "from langchain import OpenAI\n",
                "import numpy as np"
            ]
        },
        {
            "cell_type": "code", 
            "source": [
                "# Helper function to clean text\n",
                "def clean_text(text):\n",
                "    \"\"\"Remove extra whitespace and normalize text.\"\"\"\n",
                "    return ' '.join(text.split())\n",
                "\n",
                "# Another helper\n",
                "def tokenize(text):\n",
                "    \"\"\"Simple tokenization.\"\"\"\n",
                "    return text.lower().split()"
            ]
        },
        {
            "cell_type": "code",
            "source": [
                "class VectorStore:\n",
                "    \"\"\"Simple vector storage for embeddings.\"\"\"\n",
                "    def __init__(self):\n",
                "        self.vectors = []\n",
                "    \n",
                "    def add(self, vector):\n",
                "        self.vectors.append(vector)\n",
                "    \n",
                "    def search(self, query_vector, k=5):\n",
                "        # Simplified search\n",
                "        return self.vectors[:k]"
            ]
        },
        {
            "cell_type": "code",
            "source": [
                "def build_rag_graph(documents):\n",
                "    \"\"\"Main function to build RAG pipeline.\"\"\"\n",
                "    store = VectorStore()\n",
                "    \n",
                "    for doc in documents:\n",
                "        cleaned = clean_text(doc)\n",
                "        tokens = tokenize(cleaned)\n",
                "        # Add to store (simplified)\n",
                "        store.add(tokens)\n",
                "    \n",
                "    return store\n",
                "\n",
                "# Configuration\n",
                "MAX_TOKENS = 100\n",
                "TEMPERATURE = 0.7"
            ]
        }
    ],
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python", 
            "name": "python3"
        }
    }
}

print("Created tiny demo notebook with", len(tiny_demo_notebook["cells"]), "cells")

Created tiny demo notebook with 5 cells


In [5]:
# Test the NotebookParser on our tiny demo
parser = NotebookParser()
graph_data = parser.parse_notebook(tiny_demo_notebook)

print(f"Found {len(graph_data.nodes)} nodes and {len(graph_data.edges)} edges\n")

# Display all nodes
print("Nodes found:")
for node in graph_data.nodes:
    if node.type != 'cell':  # Skip cell nodes for cleaner output
        print(f"  {node.type.upper()}: {node.name} (Cell {node.cell_index + 1}, Line {node.line_in_cell})")
        if node.dependencies:
            print(f"    Dependencies: {', '.join(node.dependencies)}")

print("\nEdges (dependencies):")
for edge in graph_data.edges:
    source_node = next(n for n in graph_data.nodes if n.id == edge['source'])
    target_node = next(n for n in graph_data.nodes if n.id == edge['target'])
    print(f"  {source_node.name} -> {target_node.name} ({edge['type']})")

Found 16 nodes and 0 edges

Nodes found:
  IMPORT: pandas as pd (Cell 2, Line 1)
  IMPORT: langchain (Cell 2, Line 2)
  IMPORT: numpy as np (Cell 2, Line 3)
  FUNCTION: clean_text (Cell 3, Line 1)
  FUNCTION: tokenize (Cell 3, Line 6)
  CLASS: VectorStore (Cell 4, Line 0)
  FUNCTION: __init__ (Cell 4, Line 2)
  FUNCTION: add (Cell 4, Line 5)
  FUNCTION: search (Cell 4, Line 8)
  FUNCTION: build_rag_graph (Cell 5, Line 0)
  VARIABLE: MAX_TOKENS (Cell 5, Line 13)
  VARIABLE: TEMPERATURE (Cell 5, Line 14)

Edges (dependencies):


## Task 2: Structured Explanation

Generate a structured initial explanation of the code, ordered in a way whose explanation is best for a beginner

Creates "phases" or "chunks" of the application (if applicable)

Now let's implement the key components from chatgpt.md: caching, phase planning, and the initial explanation structure

In [14]:
INITIAL_EXPLANATION_STRUCTURE = """You are an expert Python tutor who just finished writing the attached notebook / script.

**Goal:** Produce an *initial*, learner-friendly walkthrough so that a reader with basic Python skills can eventually understand what every cell or file fragment does and how the pieces fit together.

### How to structure the explanation  
1. **High-level overview**  
   • What the program achieves and why someone would run it.  
2. **Phase map** (0-N phases)  
   • Each *phase* is a coherent chunk of work (e.g., *Data Loading*, *Helper Functions*, *Model Training*).  
   • Give each phase ✔ a short prose summary (1-3 sentences) and ✔ a numbered handle so we can refer back to it.  
3. **Phase details** - ordered for comprehension, not file order  
   For each phase  
   1. *What* it does in 2-4 beginner-level sentences.  
   2. *How* it works: describe the key components **only at the level needed for a first pass** (e.g., "calls the helper function `clean_text`, then groups rows by author").  
   3. List *components* referenced in this phase (functions, classes, imports, top-level variables) as a Markdown bullet list:  
      ```
      - def clean_text(): …
      - class VectorStore(): …
      - import langchain …
      ```
      Do **not** explain these components yet – just enumerate them so we can hook detailed tooltips later.  

### Writing style
* • *Teach, don't lecture*: assume curiosity but limited prior knowledge.  
* • Skip microscopic details (e.g., how `sum()` works) in this pass; simply note "adds values" so we can attach drill-down tooltips later.  
* • Keep paragraphs short; prefer lists where natural.
"""

# Caching implementation based on chatgpt.md
import sqlite3
from datetime import datetime

class NotebookCache:
    """Cache implementation for notebook parsing and explanations."""
    
    def __init__(self, db_path='ember_cache.db'):
        self.conn = sqlite3.connect(db_path)
        self._create_tables()
    
    def _create_tables(self):
        """Create cache tables if they don't exist."""
        self.conn.executescript('''
            CREATE TABLE IF NOT EXISTS file_state (
                hash TEXT PRIMARY KEY,
                path TEXT,
                mtime REAL,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );
            
            CREATE TABLE IF NOT EXISTS code_graph (
                file_hash TEXT PRIMARY KEY,
                graph_json TEXT,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );
            
            CREATE TABLE IF NOT EXISTS phase_plan (
                cache_key TEXT PRIMARY KEY,  -- (file_hash, prompt_version)
                phases_json TEXT,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );
            
            CREATE TABLE IF NOT EXISTS component_tooltips (
                cache_key TEXT PRIMARY KEY,  -- (component_name, file_hash)
                tooltip TEXT,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );
            
            CREATE TABLE IF NOT EXISTS external_docs (
                cache_key TEXT PRIMARY KEY,  -- (package, version)
                doc_json TEXT,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );
        ''')
        self.conn.commit()
    
    def get_file_hash(self, content: str) -> str:
        """Generate SHA-256 hash of file content."""
        return hashlib.sha256(content.encode()).hexdigest()
    
    def get_code_graph(self, file_hash: str) -> Optional[Dict[str, Any]]:
        """Retrieve cached code graph."""
        cursor = self.conn.execute(
            'SELECT graph_json FROM code_graph WHERE file_hash = ?', 
            (file_hash,)
        )
        row = cursor.fetchone()
        return json.loads(row[0]) if row else None
    
    def save_code_graph(self, file_hash: str, graph_data: GraphData):
        """Save code graph to cache."""
        graph_json = json.dumps(graph_to_dict(graph_data))
        self.conn.execute(
            'INSERT OR REPLACE INTO code_graph (file_hash, graph_json) VALUES (?, ?)',
            (file_hash, graph_json)
        )
        self.conn.commit()
    
    def get_phase_plan(self, file_hash: str, prompt_version: str) -> Optional[Dict[str, Any]]:
        """Retrieve cached phase plan."""
        cache_key = f"{file_hash}:{prompt_version}"
        cursor = self.conn.execute(
            'SELECT phases_json FROM phase_plan WHERE cache_key = ?',
            (cache_key,)
        )
        row = cursor.fetchone()
        return json.loads(row[0]) if row else None
    
    def save_phase_plan(self, file_hash: str, prompt_version: str, phases: Dict[str, Any]):
        """Save phase plan to cache."""
        cache_key = f"{file_hash}:{prompt_version}"
        self.conn.execute(
            'INSERT OR REPLACE INTO phase_plan (cache_key, phases_json) VALUES (?, ?)',
            (cache_key, json.dumps(phases))
        )
        self.conn.commit()
    
    def close(self):
        """Close database connection."""
        self.conn.close()


# Test caching with our tiny demo
cache = NotebookCache()
notebook_content = json.dumps(tiny_demo_notebook)
file_hash = cache.get_file_hash(notebook_content)

print(f"Notebook hash: {file_hash[:16]}...")

# Cache the graph data
cache.save_code_graph(file_hash, graph_data)
cached_graph = cache.get_code_graph(file_hash)
print(f"Cached graph retrieved: {cached_graph is not None}")

cache.close()

Notebook hash: cd749ff846a24125...
Cached graph retrieved: True


## Task 3: State Management for Walkthrough Session

Implementing the state machines suggested in chatgpt.md

In [19]:
@dataclass
class Phase:
    """Represents a phase in the code explanation."""
    phase_id: str
    title: str
    summary: str
    details: Optional[str] = None
    components: List[str] = field(default_factory=list)
    

@dataclass 
class ComponentInfo:
    """Extended information for a code component."""
    node_id: str
    name: str
    type: str
    phase_ids: List[str] = field(default_factory=list)
    tooltip: Optional[str] = None
    source_link: Optional[str] = None
    

@dataclass
class WalkthroughSession:
    """Tracks user progress through code walkthrough."""
    session_id: str
    file_hash: str
    current_phase: Optional[str] = None
    current_component: Optional[str] = None
    visited_phases: List[str] = field(default_factory=list)
    visited_components: List[str] = field(default_factory=list)
    
class NotebookExplainerV2(NotebookExplainer):
    """Enhanced explainer that uses real parsed data."""
    
    def _generate_phase_plan_from_graph(self, graph_data: GraphData) -> List[Phase]:
        """Generate phases based on actual parsed graph data."""
        phases = []
        
        # Group components by type and cell
        imports_by_cell = {}
        functions_by_cell = {}
        classes_by_cell = {}
        variables_by_cell = {}
        
        for node in graph_data.nodes:
            if node.type == 'import':
                imports_by_cell.setdefault(node.cell_index, []).append(node.name)
            elif node.type == 'function':
                functions_by_cell.setdefault(node.cell_index, []).append(node.name)
            elif node.type == 'class':
                classes_by_cell.setdefault(node.cell_index, []).append(node.name)
            elif node.type == 'variable':
                variables_by_cell.setdefault(node.cell_index, []).append(node.name)
        
        # Phase 1: Setup and Imports (if any imports exist)
        if imports_by_cell:
            import_components = []
            for cell_imports in imports_by_cell.values():
                import_components.extend(cell_imports)
            
            phases.append(Phase(
                phase_id="p1",
                title="Setup and Imports",
                summary="Import necessary libraries and set up the environment.",
                components=import_components
            ))
        
        # Phase 2: Helper Functions (if any exist)
        helper_functions = []
        for cell_idx, funcs in functions_by_cell.items():
            # Consider functions that don't call other functions as helpers
            for func in funcs:
                func_node = next(n for n in graph_data.nodes if n.name == func and n.type == 'function')
                if len(func_node.dependencies) == 0 or all(
                    dep in ['print', 'len', 'join', 'split', 'lower', 'append'] 
                    for dep in func_node.dependencies
                ):
                    helper_functions.append(func)
        
        if helper_functions:
            phases.append(Phase(
                phase_id="p2",
                title="Helper Functions",
                summary="Utility functions that provide basic functionality used by other components.",
                components=helper_functions
            ))
        
        # Phase 3: Core Components (classes and main functions)
        core_components = []
        
        # Add all classes
        for cell_classes in classes_by_cell.values():
            core_components.extend(cell_classes)
        
        # Add functions that call other custom functions
        for cell_idx, funcs in functions_by_cell.items():
            for func in funcs:
                if func not in helper_functions:
                    core_components.append(func)
        
        # Add important variables
        for cell_vars in variables_by_cell.values():
            core_components.extend(cell_vars)
        
        if core_components:
            phases.append(Phase(
                phase_id="p3",
                title="Core Components",
                summary="Main classes, functions, and configuration that implement the core functionality.",
                components=core_components
            ))
        
        return phases
    
    def assign_components_to_phases_from_graph(self, graph_data: GraphData, phases: List[Phase]) -> Dict[str, ComponentInfo]:
        """Assign components to phases based on actual graph data."""
        component_registry = {}
        
        # Create a mapping of component names to phase IDs
        component_to_phases = {}
        for phase in phases:
            for component_name in phase.components:
                if component_name not in component_to_phases:
                    component_to_phases[component_name] = []
                component_to_phases[component_name].append(phase.phase_id)
        
        # Create ComponentInfo for each node
        for node in graph_data.nodes:
            if node.type != 'cell':
                phase_ids = component_to_phases.get(node.name, [])
                
                # Generate tooltip based on node type and content
                tooltip = self._generate_tooltip(node)
                
                component_info = ComponentInfo(
                    node_id=node.id,
                    name=node.name,
                    type=node.type,
                    phase_ids=phase_ids,
                    tooltip=tooltip,
                    source_link=f"cell_{node.cell_index}#L{node.line_in_cell}"
                )
                component_registry[node.name] = component_info
        
        return component_registry
    
    def _generate_tooltip(self, node: CodeNode) -> str:
        """Generate a tooltip description for a component."""
        if node.type == 'import':
            return f"Imports {node.name} module"
        elif node.type == 'function':
            if node.dependencies:
                return f"Function that calls: {', '.join(node.dependencies)}"
            else:
                return "Utility function"
        elif node.type == 'class':
            return f"Class definition with {len(node.children)} methods"
        elif node.type == 'variable':
            return "Configuration variable"
        return "Code component"
    
    def analyze_notebook(self, notebook_data: Dict[str, Any]) -> Tuple[GraphData, List[Phase]]:
        """Override to use real data generation."""
        # Get file hash
        notebook_content = json.dumps(notebook_data)
        file_hash = self.cache.get_file_hash(notebook_content)
        
        # Parse notebook (using cache if available)
        cached_graph_dict = self.cache.get_code_graph(file_hash)
        if cached_graph_dict:
            graph_data = self._dict_to_graph(cached_graph_dict)
            print("Using cached code graph")
        else:
            graph_data = self.parser.parse_notebook(notebook_data)
            self.cache.save_code_graph(file_hash, graph_data)
            print("Parsed and cached new code graph")
        
        # Generate phases from actual data
        phases = self._generate_phase_plan_from_graph(graph_data)
        
        return graph_data, phases


# Test with the enhanced explainer
print("=== Testing Enhanced Explainer with Real Data ===\n")
explainer_v2 = NotebookExplainerV2(openai_client=openai_client)
graph_data, phases = explainer_v2.analyze_notebook(tiny_demo_notebook)

print("Generated Phases from Actual Data:")
for phase in phases:
    print(f"\n{phase.phase_id}: {phase.title}")
    print(f"  Summary: {phase.summary}")
    print(f"  Components: {', '.join(phase.components)}")

# Assign components with real data
component_registry = explainer_v2.assign_components_to_phases_from_graph(graph_data, phases)

print(f"\n\nComponent Registry ({len(component_registry)} components):")
for name, info in component_registry.items():
    print(f"\n{name} ({info.type}):")
    print(f"  Phases: {', '.join(info.phase_ids)}")
    print(f"  Tooltip: {info.tooltip}")
    print(f"  Source: {info.source_link}")

explainer_v2.close()

=== Testing Enhanced Explainer with Real Data ===

Using cached code graph
Generated Phases from Actual Data:

p1: Setup and Imports
  Summary: Import necessary libraries and set up the environment.
  Components: pandas as pd, langchain, numpy as np

p2: Helper Functions
  Summary: Utility functions that provide basic functionality used by other components.
  Components: clean_text, tokenize, __init__, add, search, build_rag_graph

p3: Core Components
  Summary: Main classes, functions, and configuration that implement the core functionality.
  Components: VectorStore, MAX_TOKENS, TEMPERATURE


Component Registry (12 components):

pandas as pd (import):
  Phases: p1
  Tooltip: Imports pandas as pd module
  Source: cell_1#L1

langchain (import):
  Phases: p1
  Tooltip: Imports langchain module
  Source: cell_1#L2

numpy as np (import):
  Phases: p1
  Tooltip: Imports numpy as np module
  Source: cell_1#L3

clean_text (function):
  Phases: p2
  Tooltip: Utility function
  Source: cell_2#

## Walkthrough Session Demo

Let's create a walkthrough session to demonstrate how users would navigate through the code

In [17]:
import uuid

class WalkthroughManager:
    """Manages user walkthrough sessions."""
    
    def __init__(self, graph_data: GraphData, phases: List[Phase], component_registry: Dict[str, ComponentInfo]):
        self.graph_data = graph_data
        self.phases = phases
        self.component_registry = component_registry
        self.phase_order = [p.phase_id for p in phases]
        
    def create_session(self, file_hash: str) -> WalkthroughSession:
        """Create a new walkthrough session."""
        return WalkthroughSession(
            session_id=str(uuid.uuid4()),
            file_hash=file_hash,
            current_phase=self.phases[0].phase_id if self.phases else None
        )
    
    def get_current_content(self, session: WalkthroughSession) -> Dict[str, Any]:
        """Get content for the current position in the walkthrough."""
        if not session.current_phase:
            return {"type": "overview", "content": "No phases available"}
        
        current_phase = next(p for p in self.phases if p.phase_id == session.current_phase)
        
        # Get components for this phase
        phase_components = []
        for comp_name in current_phase.components:
            if comp_name in self.component_registry:
                comp_info = self.component_registry[comp_name]
                # Find the actual node
                node = next((n for n in self.graph_data.nodes if n.name == comp_name and n.type != 'cell'), None)
                if node:
                    phase_components.append({
                        "name": comp_name,
                        "type": comp_info.type,
                        "tooltip": comp_info.tooltip,
                        "content": node.content,
                        "cell": node.cell_index + 1,
                        "line": node.line_in_cell + 1
                    })
        
        return {
            "type": "phase",
            "phase_id": current_phase.phase_id,
            "title": current_phase.title,
            "summary": current_phase.summary,
            "components": phase_components,
            "progress": {
                "current": self.phase_order.index(session.current_phase) + 1,
                "total": len(self.phases)
            }
        }
    
    def next_phase(self, session: WalkthroughSession) -> bool:
        """Move to the next phase. Returns True if successful."""
        if not session.current_phase:
            return False
            
        current_idx = self.phase_order.index(session.current_phase)
        if current_idx < len(self.phase_order) - 1:
            session.visited_phases.append(session.current_phase)
            session.current_phase = self.phase_order[current_idx + 1]
            return True
        return False
    
    def previous_phase(self, session: WalkthroughSession) -> bool:
        """Move to the previous phase. Returns True if successful."""
        if not session.current_phase:
            return False
            
        current_idx = self.phase_order.index(session.current_phase)
        if current_idx > 0:
            session.current_phase = self.phase_order[current_idx - 1]
            return True
        return False


# Create a walkthrough demo
print("=== Walkthrough Session Demo ===\n")

# Initialize walkthrough manager
manager = WalkthroughManager(graph_data, phases, component_registry)

# Create a session
notebook_content = json.dumps(tiny_demo_notebook)
file_hash = hashlib.sha256(notebook_content.encode()).hexdigest()
session = manager.create_session(file_hash)

print(f"Created session: {session.session_id[:8]}...")
print(f"Starting at phase: {session.current_phase}\n")

# Walk through each phase
while True:
    content = manager.get_current_content(session)
    
    print(f"\n{'='*60}")
    print(f"Phase {content['progress']['current']}/{content['progress']['total']}: {content['title']}")
    print(f"{'='*60}")
    print(f"\nSummary: {content['summary']}\n")
    
    if content['components']:
        print("Components in this phase:")
        for comp in content['components']:
            print(f"\n  📄 {comp['name']} ({comp['type']})")
            print(f"     Location: Cell {comp['cell']}, Line {comp['line']}")
            print(f"     Tooltip: {comp['tooltip']}")
            print(f"     Code: {comp['content'][:50]}...")
    
    # Try to go to next phase
    if not manager.next_phase(session):
        print("\n✅ Walkthrough complete!")
        break

print(f"\n\nVisited phases: {session.visited_phases}")

=== Walkthrough Session Demo ===

Created session: 33c6b892...
Starting at phase: p1


Phase 1/3: Setup and Imports

Summary: Import necessary libraries and set up the environment.

Components in this phase:

  📄 pandas as pd (import)
     Location: Cell 2, Line 2
     Tooltip: Imports pandas as pd module
     Code: import pandas as pd...

  📄 langchain (import)
     Location: Cell 2, Line 3
     Tooltip: Imports langchain module
     Code: from langchain import OpenAI...

  📄 numpy as np (import)
     Location: Cell 2, Line 4
     Tooltip: Imports numpy as np module
     Code: import numpy as np...

Phase 2/3: Helper Functions

Summary: Utility functions that provide basic functionality used by other components.

Components in this phase:

  📄 clean_text (function)
     Location: Cell 3, Line 2
     Tooltip: Utility function
     Code: def clean_text(text):...

  📄 tokenize (function)
     Location: Cell 3, Line 7
     Tooltip: Utility function
     Code: def tokenize(text):...

  📄 _

## Dependency Visualization

Let's visualize the dependencies found in our tiny notebook

In [18]:
# Show dependency graph
print("=== Dependency Graph ===\n")

# First, let's see all the nodes we found
print("All Components:")
for node in graph_data.nodes:
    if node.type != 'cell':
        print(f"  {node.name} ({node.type})")
        if node.dependencies:
            print(f"    → Calls: {', '.join(node.dependencies)}")

print("\n\nDependency Edges:")
if graph_data.edges:
    for edge in graph_data.edges:
        source_node = next(n for n in graph_data.nodes if n.id == edge['source'])
        target_node = next(n for n in graph_data.nodes if n.id == edge['target'])
        print(f"  {source_node.name} --{edge['type']}--> {target_node.name}")
else:
    print("  No dependencies detected between custom components")

# Let's check why dependencies might not be detected
print("\n\nDetailed Function Analysis:")
for node in graph_data.nodes:
    if node.type == 'function':
        print(f"\n{node.name}:")
        print(f"  Full content: {node.content}")
        
        # Look for function calls in the entire cell content
        cell_node = next(n for n in graph_data.nodes if n.type == 'cell' and n.cell_index == node.cell_index)
        cell_lines = cell_node.content.split('\n')
        
        # Find where this function starts and ends
        func_start = node.line_in_cell
        func_end = func_start
        
        # Find the end of the function by looking for the next non-indented line
        for i in range(func_start + 1, len(cell_lines)):
            line = cell_lines[i]
            if line and not line.startswith((' ', '\t')):
                func_end = i - 1
                break
            elif i == len(cell_lines) - 1:
                func_end = i
        
        # Extract function body
        func_body = '\n'.join(cell_lines[func_start:func_end + 1])
        print(f"  Function body:\n{func_body}")

=== Dependency Graph ===

All Components:
  pandas as pd (import)
  langchain (import)
  numpy as np (import)
  clean_text (function)
  tokenize (function)
  VectorStore (class)
  __init__ (function)
  add (function)
  search (function)
  build_rag_graph (function)
  MAX_TOKENS (variable)
  TEMPERATURE (variable)


Dependency Edges:
  No dependencies detected between custom components


Detailed Function Analysis:

clean_text:
  Full content: def clean_text(text):
  Function body:
def clean_text(text):
    """Remove extra whitespace and normalize text."""
    return ' '.join(text.split())


tokenize:
  Full content: def tokenize(text):
  Function body:
def tokenize(text):
    """Simple tokenization."""
    return text.lower().split()

__init__:
  Full content: def __init__(self):
  Function body:
    def __init__(self):
        self.vectors = []
    
    def add(self, vector):
        self.vectors.append(vector)
    
    def search(self, query_vector, k=5):
        # Simplified search


## Code Assignment

(Ask ChatGPT if this is best done simultaneously or separate)

Pass through initial explanation and parsed content and have LLM assign to one or more of the phases. 

## Phase Breakdown

At each phase, subsection and refine the phase given the phase's initial explanation and list of parsed content. 

Parsed content gets a saved brief description of what it does in the application. 

Before refining, will first check to see if all imports and functions are accounted for in meaning. If certain libraries are not known, it will do a search for the library doc (making sure the version is the same), save description of library overview and link, as well as descriptions for all used functions, etc.

If a *called* function description it needs is from another phase that has not finished its description, it waits for it to return before 


Returns: 
1) Refined phase covering all code
2) Descriptions for all parsed content
3) Phase state updated with new content, a list of subsections, and those subsection state containing their individual explanations and parsed content in sequential order


New phase state will now contain only an overview instead of full explanation (if applicable). Imports can be phase-level or subsection-level depending on granularity of usage (if used across multiple subsections, then phase-level)

Parsed content state can have parent/child fields as well as call/called-by fields

## LLM Interaction

If any library or function is brought up or asked about, first checks saved documentation from search to see if there is any related content, if not, then uses inherent knowledge, before finally using search if needed.

Returns:
1) Answer
2) Stores answer in adjacent line

Why?

Myself and my classmates would often ask what a piece of code does or how it fits in.

I love using ChatGPT, Grok, and Claude for code explanations, but I find I often need to refer back to explanations, especially as my questions accumulate, usually scrolling for some bit to look for exactly where I found it.

Future:

- Caching
- Bidirectional edges (specify called vs called by)
- Have LangGraph grouped/highlighted and labeled?
- - May need to modify initial prompt to make sure that if a LangGraph is buildable, they should all connect

- Locking ipynb to prevent change unless in Editing Mode
- LLM Responses reference actual parsed content, which makes it clickable and then highlights them in the graph and editor
- Restructure or make new ipynb in the image of the initial explanation's structure order
- - Less jumping around
- - Allows more intuitive/coherent visual modification (below: LLM Builder)
- Per section interactive variable modification
- - Does not change code during interactive runs
- - Can write individual changed variables to code or all
- - Can save "x" amount of past variable runs and their results
- - Can "hide" variables if not planned to be changeable variables
- - Includes what it does and what/how it might affect application
- LLM Builder/Editing Mode
- - Create visual phase and subphase groups along with nodes 
- Extend to .py and modules

Tracked Hours:

7/30
6am-7:30am     1.5h

In [ ]:
#   Proposed Iterative Improvement Architecture

#   INITIAL ANALYSIS
#         |
#         v
#   DEEP DESCRIPTIONS
#         |
#         v
#   IMPROVEMENT ASSESSMENT (Judging Agent)
#         |
#         +---> <10% improvement potential? --> DONE
#         |
#         +---> ≥10% improvement potential? --> SELECTIVE ITERATION
#                                                       |
#                                                       v
#                                                 UPDATE STRATEGY